# Nucleic Acid Structure

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Describe the chemical components of nucleotides (base, sugar, phosphate)
- Distinguish DNA helical forms (A, B, Z) and their biological significance
- Explain major and minor groove geometry and its role in protein-DNA recognition
- Understand RNA secondary structure elements (stems, loops, bulges, junctions)
- Conceptually understand RNA secondary structure prediction via minimum free energy
- Analyze DNA-protein interactions: transcription factors, nucleosomes
- Parse nucleic acid structures from PDB files with BioPython
- Use tools for nucleic acid visualization and analysis

## Why this notebook matters

Understanding nucleic acid structure is foundational for interpreting sequencing data, designing primers, predicting RNA secondary structure, and understanding gene regulation. Watson-Crick base pairing, helix geometry, major/minor grooves, RNA secondary structure, and DNA-protein interactions all directly impact how bioinformatics algorithms work. This notebook covers DNA helix forms (A, B, Z), RNA secondary structure prediction (dot-bracket notation, Nussinov algorithm), restriction site analysis, and parsing nucleic acid structures from PDB files.

## How to use this notebook

1. Run cells top-to-bottom in order — later cells depend on earlier ones
2. Run the environment check cell first to confirm all imports work
3. Read the explanatory text before each code cell — the context matters
4. The exercises at the end are designed to be done after reading each section
5. If a code cell requires internet access (Entrez, PDB download), it is marked — these can be skipped if offline

In [ ]:
## Complicated moments explained

- **DNA strand polarity matters everywhere**: DNA is always synthesized 5'→3'. The two strands in a duplex run antiparallel — if one strand runs 5'→3' left to right, the complementary strand runs 3'→5' left to right (or equivalently, 5'→3' right to left). When searching for motifs, always search both strands.
- **B-DNA vs. A-DNA vs. Z-DNA**: B-DNA is the canonical right-handed helix under physiological conditions (~10.5 bp/turn, 3.4 Å rise/bp). A-DNA forms in dehydrated conditions (~11 bp/turn, 2.6 Å rise/bp, wider and shorter). Z-DNA is left-handed (~12 bp/turn) and forms in alternating purine-pyrimidine sequences under high salt or torsional stress. Most genomic DNA is B-form.
- **RNA secondary structure complexity**: Unlike DNA, RNA is predominantly single-stranded and folds into complex secondary and tertiary structures via intramolecular base pairing. The same RNA sequence can fold multiple ways; minimum free energy (MFE) structure is only one possibility. Use ViennaRNA or RNAfold for MFE prediction; consider suboptimal structures for regulatory RNAs.
- **Dot-bracket notation**: In dot-bracket notation, `(` and `)` mark paired bases (the `(` at position i pairs with the `)` at position j), and `.` marks unpaired bases. Pseudoknots require `[`, `]`, `{`, `}` brackets and many tools cannot represent them.
- **CpG islands**: Regions with CpG O/E ratio > 0.6, GC content > 55%, and length > 200 bp are often promoters of housekeeping genes. Most CpG dinucleotides in mammalian genomes are methylated and have been depleted by deamination (5mC → T mutation).

## Environment check (run this first)

In [ ]:
# Environment check
from Bio.Seq import Seq
from Bio.PDB import PDBParser, PDBList
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Imports ready.")

# Quick demo: base pairing and complement
dna = Seq("ATGCGAATTCGATCG")
print(f"Sequence (5'->3'):     {dna}")
print(f"Complement (3'->5'):   {dna.complement()}")
print(f"Rev. complement (5'->3'): {dna.reverse_complement()}")
print()

# Helix parameters at a glance
helix_params = {
    'B-DNA': {'bp_per_turn': 10.5, 'rise_A': 3.4, 'handedness': 'Right', 'diameter_A': 20},
    'A-DNA': {'bp_per_turn': 11.0, 'rise_A': 2.6, 'handedness': 'Right', 'diameter_A': 23},
    'Z-DNA': {'bp_per_turn': 12.0, 'rise_A': 3.7, 'handedness': 'Left',  'diameter_A': 18},
}
print(f"{'Form':<8} {'bp/turn':<10} {'Rise(Å)':<10} {'Diameter(Å)':<14} {'Handedness'}")
print("-" * 55)
for form, p in helix_params.items():
    print(f"{form:<8} {p['bp_per_turn']:<10} {p['rise_A']:<10} {p['diameter_A']:<14} {p['handedness']}")
print("\nProceed to Section 1.")

---

## 3. DNA Double Helix: A, B, and Z Forms

DNA can adopt different helical conformations depending on sequence, hydration, and salt concentration:

```
A-DNA                   B-DNA                    Z-DNA
(dehydrated)            (physiological)          (high salt, alt. pur-pyr)

  /\                      /\                       /\
 /  \                    /  \                     /  \
| ** |  wider,          |    |  the standard     | zz |  zig-zag
 \ **/  shorter         |    |  Watson-Crick      \zz/   backbone
  |  |                  |    |  helix              |  |
  | **|                 |    |                     |zz|
  \** /                  \  /                      \  /
   \/                    \/                        \/

Right-handed            Right-handed              Left-handed
```

| Parameter | A-DNA | B-DNA | Z-DNA |
|-----------|-------|-------|-------|
| Handedness | Right | Right | **Left** |
| Base pairs / turn | 11 | 10.5 | 12 |
| Rise / base pair | 2.6 A | 3.4 A | 3.7 A |
| Helix diameter | 23 A | 20 A | 18 A |
| Major groove | Narrow, deep | Wide, deep | Flat |
| Minor groove | Wide, shallow | Narrow, deep | Narrow, deep |
| Sugar pucker | C3'-endo | C2'-endo | Alternating |
| Conditions | Dehydrated, RNA-DNA hybrids | Physiological | High salt, alternating purine-pyrimidine |

**B-DNA** is the standard physiological form. **A-form** is adopted by RNA duplexes and DNA-RNA hybrids (the 2'-OH of RNA forces C3'-endo sugar pucker). **Z-DNA** forms in regions with alternating purine-pyrimidine sequences under high salt conditions; it may play roles in gene regulation.

In [ ]:
# DNA helix parameters and dimension calculations
HELIX_PARAMS = {
    'A-DNA': {'bp_per_turn': 11,   'rise': 2.6, 'diameter': 23, 'handedness': 'Right'},
    'B-DNA': {'bp_per_turn': 10.5, 'rise': 3.4, 'diameter': 20, 'handedness': 'Right'},
    'Z-DNA': {'bp_per_turn': 12,   'rise': 3.7, 'diameter': 18, 'handedness': 'Left'},
}


def helix_dimensions(num_bp, form='B-DNA'):
    """
    Calculate helix dimensions for a given number of base pairs.
    
    Returns dict with length (A and nm), number of turns, diameter.
    """
    p = HELIX_PARAMS[form]
    length_A = num_bp * p['rise']
    return {
        'form': form,
        'base_pairs': num_bp,
        'length_A': length_A,
        'length_nm': length_A / 10.0,
        'turns': num_bp / p['bp_per_turn'],
        'diameter_A': p['diameter']
    }


# Compare 100 bp in different forms
print("100 bp DNA in different helical forms:\n")
print(f"{'Form':<8s} {'Length (nm)':>12s} {'Turns':>8s} {'Diameter (A)':>13s}")
print("-" * 45)
for form in HELIX_PARAMS:
    d = helix_dimensions(100, form)
    print(f"{d['form']:<8s} {d['length_nm']:12.1f} {d['turns']:8.1f} {d['diameter_A']:13.0f}")

# Human genome scale
print("\n--- Human Genome Scale ---")
genome_bp = 3.2e9  # ~3.2 billion base pairs
d = helix_dimensions(genome_bp, 'B-DNA')
print(f"Human genome ({genome_bp/1e9:.1f} Gbp) as B-DNA:")
print(f"  Length: {d['length_nm']/1e9:.2f} m = {d['length_nm']/1e9*100:.0f} cm")
print(f"  Turns:  {d['turns']/1e6:.0f} million")
print(f"  (That is ~2 meters of DNA packed into a ~6 um nucleus!)")

---

## 4. Major and Minor Grooves

The double helix has two grooves of unequal size, created because the glycosidic bonds of a base pair are not diametrically opposite:

```
                  MAJOR GROOVE (~12 A wide in B-DNA)
                /                          \
   5' --------/  Rich in chemical groups    \-------- 3'
      \      |   for protein recognition    |      /
       \     |                              |     /
        \----+--------- base pairs ---------+----/
       /     |                              |     \
      /      |   Fewer distinguishing       |      \
   3' --------\  chemical features          /-------- 5'
                \                          /
                  MINOR GROOVE (~6 A wide in B-DNA)


Groove accessibility in different forms:

         Major groove    Minor groove
A-DNA    Narrow, deep    Wide, shallow
B-DNA    Wide, deep      Narrow, deep
Z-DNA    Flat            Narrow, deep
```

### Why Grooves Matter

The **major groove** exposes a unique pattern of hydrogen bond donors and acceptors for each base pair, allowing proteins to "read" the DNA sequence without unwinding the helix:

| Base Pair | Major Groove Pattern | Minor Groove Pattern |
|-----------|---------------------|---------------------|
| A-T | H-bond acceptor, no group, H-bond acceptor, H, CH3 | H-bond acceptor, no group |
| T-A | CH3, H, H-bond acceptor, no group, H-bond acceptor | no group, H-bond acceptor |
| G-C | H-bond acceptor, H-bond acceptor, H-bond donor, no group | H-bond acceptor, H-bond donor |
| C-G | no group, H-bond donor, H-bond acceptor, H-bond acceptor | H-bond donor, H-bond acceptor |

All four base pairs are distinguishable in the **major groove** but A-T vs T-A (and G-C vs C-G) are hard to tell apart in the minor groove. This is why most transcription factors bind in the major groove.

### Base Stacking

In addition to hydrogen bonding between bases, **stacking interactions** between adjacent base pairs are a major stabilizing force. These arise from:

- Van der Waals forces between the flat aromatic rings
- Dipole-dipole interactions
- Hydrophobic effect (excluding water from the helix interior)

Stacking contributes more to duplex stability than hydrogen bonding. The stacking energy depends on the identity of adjacent base pairs (nearest-neighbor model).

In [ ]:
# Nearest-neighbor stacking free energies (kcal/mol) at 37C, 1M NaCl
# From SantaLucia (1998)
NN_ENERGIES = {
    'AA': -1.0, 'AT': -0.88, 'AG': -1.28, 'AC': -1.44,
    'TA': -0.58, 'TT': -1.0,  'TG': -1.45, 'TC': -1.30,
    'GA': -1.30, 'GT': -1.44, 'GG': -1.84, 'GC': -2.24,
    'CA': -1.45, 'CT': -1.28, 'CG': -2.17, 'CC': -1.84,
}


def nearest_neighbor_dG(seq):
    """
    Estimate duplex free energy using the nearest-neighbor model.
    
    More accurate than simple base-pair counting.
    Initiation penalty: +1.96 kcal/mol (for both ends)
    """
    seq = seq.upper()
    dG = 1.96  # Initiation penalty
    
    for i in range(len(seq) - 1):
        dinuc = seq[i:i+2]
        dG += NN_ENERGIES.get(dinuc, -1.0)
    
    return dG


# Compare stacking energies for different sequences
test_seqs = [
    'GCGCGCGC',   # Alternating GC
    'GGGGCCCC',   # GC block
    'AAAATTTT',   # AT block
    'ATATATAT',   # Alternating AT
    'GATCGATC',   # Mixed
]

print(f"{'Sequence':<12s} {'GC%':>5s} {'dG (kcal/mol)':>14s} {'Tm (C)':>7s}")
print("-" * 42)
for seq in test_seqs:
    dG = nearest_neighbor_dG(seq)
    tm = melting_temperature(seq)
    print(f"{seq:<12s} {gc_content(seq):5.0f} {dG:14.2f} {tm:7.0f}")

---

## 5. RNA Structure

Unlike DNA, RNA is typically single-stranded and folds back on itself to form complex secondary and tertiary structures.

### RNA Secondary Structure Elements

```
1. STEM (paired region)        2. HAIRPIN LOOP (stem-loop)

   G - C                              U  C  A
   A - U                             U      G
   C - G                            G--------C
   U - A                            |        |
   G - C                            C--------G
                                    |        |
                                    A--------U
                                    5'      3'

3. BULGE (unpaired on one side) 4. INTERNAL LOOP (unpaired on both sides)

   G - C                            G - C
   A - U                            A   U   <-- mismatches
   C   |                            G   A
   U - A  <-- C is unpaired         C - G
   G - C                            A - U

5. MULTI-STEM JUNCTION           6. PSEUDOKNOT

        G-C                         5'-A-U---G-C--\
       / | \                              |    |   |
      /  |  \                        3'-U-A---C-G--/
   G-C  A-U  C-G                    (pairs with downstream region)
   | |  | |  | |
   C-G  U-A  G-C
```

### Dot-Bracket Notation

RNA secondary structure is commonly represented in dot-bracket format:
- `.` = unpaired nucleotide
- `(` = 5' partner of a base pair
- `)` = 3' partner of a base pair

Example hairpin: `((((....))))` -- 4-bp stem with 4-nt loop

In [ ]:
def parse_dot_bracket(structure):
    """
    Parse dot-bracket notation into a list of base pairs.
    
    Returns list of (i, j) tuples where i < j (0-indexed).
    """
    pairs = []
    stack = []
    
    for i, ch in enumerate(structure):
        if ch == '(':
            stack.append(i)
        elif ch == ')':
            if not stack:
                raise ValueError(f"Unmatched ')' at position {i}")
            j = stack.pop()
            pairs.append((j, i))
    
    if stack:
        raise ValueError(f"Unmatched '(' at positions {stack}")
    
    return sorted(pairs)


def classify_structure_elements(sequence, structure):
    """
    Identify structural elements from sequence and dot-bracket structure.
    
    Returns counts of stems, hairpin loops, bulges, internal loops.
    """
    pairs = parse_dot_bracket(structure)
    pair_map = {}
    for i, j in pairs:
        pair_map[i] = j
        pair_map[j] = i
    
    n = len(structure)
    paired = set()
    for i, j in pairs:
        paired.add(i)
        paired.add(j)
    
    # Count hairpin loops: regions of unpaired bases enclosed by a base pair
    hairpin_loops = 0
    for i, j in pairs:
        # Check if all positions between i and j are unpaired
        interior = structure[i+1:j]
        if all(c == '.' for c in interior) and len(interior) > 0:
            hairpin_loops += 1
    
    return {
        'length': n,
        'base_pairs': len(pairs),
        'unpaired': n - len(paired),
        'paired_fraction': len(paired) / n,
        'hairpin_loops': hairpin_loops,
    }


# Example: a tRNA-like structure
trna_seq    = "GCGGAUUUAGCUCAGDDGGGAGAGCGCCAGACUGAAYA"
trna_struct = "((((((.....))))))....(((((......)))))..."

pairs = parse_dot_bracket(trna_struct)
stats = classify_structure_elements(trna_seq, trna_struct)

print(f"Sequence:  {trna_seq}")
print(f"Structure: {trna_struct}")
print(f"\nBase pairs: {stats['base_pairs']}")
print(f"Unpaired nucleotides: {stats['unpaired']}")
print(f"Paired fraction: {stats['paired_fraction']:.1%}")
print(f"Hairpin loops: {stats['hairpin_loops']}")
print(f"\nBase pair list: {pairs}")

In [ ]:
def visualize_dot_bracket(sequence, structure):
    """
    Create a text-based visualization of RNA secondary structure.
    Shows paired bases connected by vertical lines.
    """
    pairs = parse_dot_bracket(structure)
    pair_map = {}
    for i, j in pairs:
        pair_map[i] = j
        pair_map[j] = i
    
    # Show sequence with numbering
    n = len(sequence)
    
    # Top ruler
    ruler = ''.join(str(i % 10) for i in range(n))
    tens = ''.join(str(i // 10) if i % 10 == 0 else ' ' for i in range(n))
    
    print(f"Position:  {tens}")
    print(f"           {ruler}")
    print(f"Sequence:  {sequence}")
    print(f"Structure: {structure}")
    
    # Show base pairs as arcs (text representation)
    print(f"\nBase pairs (i -- j):")
    for i, j in pairs:
        bp = f"{sequence[i]}-{sequence[j]}"
        print(f"  {i:3d} {sequence[i]} --- {sequence[j]} {j:3d}  ({bp})")


# Visualize
hairpin_seq    = "GGCGCAAGCCUUCGCGCC"
hairpin_struct = "((((((.....))))))."
visualize_dot_bracket(hairpin_seq, hairpin_struct)

---

## 6. RNA Secondary Structure Prediction: Minimum Free Energy

RNA secondary structure can be predicted computationally using thermodynamic models.

### The Minimum Free Energy (MFE) Approach

The idea: among all possible secondary structures that an RNA sequence can form, the one with the **lowest free energy** is most likely to be the actual structure.

```
Input: RNA sequence
         |
         v
  +---------------------------+
  | Enumerate possible base   |
  | pairs and loop structures |
  +---------------------------+
         |
         v
  +---------------------------+
  | Assign free energies from |   Turner energy parameters:
  | thermodynamic parameters  |   - Stacking (base pair steps)
  +---------------------------+   - Loops (hairpin, internal, bulge)
         |                        - Dangling ends
         v                        - Coaxial stacking
  +---------------------------+
  | Dynamic programming to    |   Nussinov algorithm (simplified)
  | find the minimum free     |   Zuker algorithm (full energetics)
  | energy structure          |
  +---------------------------+
         |
         v
  Output: MFE structure (dot-bracket) + free energy
```

### Key Software

- **ViennaRNA** (RNAfold): Gold-standard RNA folding package
- **mfold**: Classic web server for RNA folding
- **RNAstructure**: Includes partition function and pseudoknot prediction

### Limitations

- Cannot predict pseudoknots (standard DP algorithm)
- Does not account for co-transcriptional folding
- Kinetic traps may prevent the MFE structure from forming
- Long-range tertiary interactions are not modeled

In [ ]:
def nussinov_fold(sequence, min_loop=3):
    """
    Simplified Nussinov algorithm for RNA secondary structure prediction.
    
    Maximizes the number of base pairs (not thermodynamically accurate,
    but illustrates the dynamic programming approach).
    
    Parameters:
        sequence: RNA sequence (AUGC)
        min_loop: Minimum loop size (default 3)
    Returns:
        Dot-bracket structure string
    """
    seq = sequence.upper()
    n = len(seq)
    
    # Valid base pairs
    can_pair = {
        ('A', 'U'), ('U', 'A'),
        ('G', 'C'), ('C', 'G'),
        ('G', 'U'), ('U', 'G'),  # Wobble pair
    }
    
    # DP table: dp[i][j] = max number of base pairs for subsequence i..j
    dp = [[0] * n for _ in range(n)]
    
    # Fill table for increasing subsequence lengths
    for length in range(min_loop + 2, n + 1):  # min_loop+2 for a pair
        for i in range(n - length + 1):
            j = i + length - 1
            
            # Option 1: j is unpaired
            dp[i][j] = dp[i][j-1]
            
            # Option 2: j pairs with some k
            for k in range(i, j - min_loop):
                if (seq[k], seq[j]) in can_pair:
                    left = dp[i][k-1] if k > i else 0
                    inside = dp[k+1][j-1] if k+1 <= j-1 else 0
                    score = left + inside + 1
                    dp[i][j] = max(dp[i][j], score)
    
    # Traceback to recover structure
    structure = ['.'] * n
    
    def traceback(i, j):
        if i >= j or dp[i][j] == 0:
            return
        if dp[i][j] == dp[i][j-1]:
            traceback(i, j-1)
        else:
            for k in range(i, j - min_loop):
                if (seq[k], seq[j]) in can_pair:
                    left = dp[i][k-1] if k > i else 0
                    inside = dp[k+1][j-1] if k+1 <= j-1 else 0
                    if left + inside + 1 == dp[i][j]:
                        structure[k] = '('
                        structure[j] = ')'
                        traceback(i, k-1)
                        traceback(k+1, j-1)
                        return
    
    traceback(0, n - 1)
    return ''.join(structure)


# Test with a hairpin-forming sequence
test_rna = "GGGAAACCC"  # Should form stem-loop: (((...)))..
predicted = nussinov_fold(test_rna)
pairs = parse_dot_bracket(predicted)

print(f"Sequence:  {test_rna}")
print(f"Predicted: {predicted}")
print(f"Base pairs: {len(pairs)}")

# A longer example
rna2 = "GGGCCCAAAGGGCCCUUU"
pred2 = nussinov_fold(rna2)
print(f"\nSequence:  {rna2}")
print(f"Predicted: {pred2}")
print(f"Base pairs: {len(parse_dot_bracket(pred2))}")

In [ ]:
# Using ViennaRNA (if installed) for proper MFE prediction
try:
    import RNA  # ViennaRNA Python bindings
    
    sequence = "GGGAAACCC"
    (structure, mfe) = RNA.fold(sequence)
    
    print(f"ViennaRNA MFE prediction:")
    print(f"  Sequence:  {sequence}")
    print(f"  Structure: {structure}")
    print(f"  MFE:       {mfe:.2f} kcal/mol")
    
    # A more complex example: part of a tRNA
    trna_seq = "GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGGUCCUGUGUUCGAUCCACAGAAUUCGCACCA"
    (trna_struct, trna_mfe) = RNA.fold(trna_seq)
    print(f"\ntRNA-like sequence:")
    print(f"  Sequence:  {trna_seq}")
    print(f"  Structure: {trna_struct}")
    print(f"  MFE:       {trna_mfe:.2f} kcal/mol")

except ImportError:
    print("ViennaRNA Python bindings not installed.")
    print("Install with: conda install -c bioconda viennarna")
    print("\nUsing our Nussinov implementation instead (less accurate):")
    
    sequence = "GGGAAACCC"
    structure = nussinov_fold(sequence)
    print(f"  Sequence:  {sequence}")
    print(f"  Structure: {structure}")

---

## 7. RNA Tertiary Interactions

Beyond secondary structure, RNA forms complex 3D structures through tertiary contacts:

### Non-Watson-Crick Base Pairs

RNA uses all three edges of its bases for pairing, as classified by the **Leontis-Westhof scheme**:

| Edge | Description |
|------|-------------|
| Watson-Crick (W) | Standard WC edge |
| Hoogsteen (H) | Major groove edge |
| Sugar (S) | Minor groove edge / 2'-OH |

Each pair can be **cis** or **trans**, giving 12 possible geometric families.

### Common Tertiary Motifs

- **A-minor interactions**: Adenine inserts into the minor groove of an adjacent helix
- **Ribose zippers**: 2'-OH hydrogen bonds between backbone riboses
- **Kissing loops**: Two hairpin loops form base pairs with each other
- **Coaxial stacking**: Two helices stack end-to-end, effectively forming one continuous helix
- **G-quadruplex**: Four guanines form a planar tetrad via Hoogsteen pairing; stacked tetrads form a quadruplex structure

### Important RNA Structures

| RNA | Function | Key Structural Features |
|-----|----------|------------------------|
| tRNA | Amino acid carrier | L-shape, D-loop, anticodon loop, CCA 3'-end |
| rRNA | Ribosome core | Extensive tertiary interactions, catalytic |
| Ribozyme | RNA catalyst | Active site formed by tertiary structure |
| Riboswitch | Gene regulation | Ligand-binding aptamer + expression platform |

---

## 8. DNA-Protein Interactions

### Transcription Factor Binding

Transcription factors (TFs) recognize specific DNA sequences, usually through the major groove:

```
Common DNA-binding motifs in proteins:

1. HELIX-TURN-HELIX (HTH)     2. ZINC FINGER
                                         Zn2+
    recognition                         / | \
    helix                             Cys Cys His
     ====                              |       |
    /    \                             ====  finger
   |      |                           / helix \
   =======                    --------         --------
  ----------                  ===== MAJOR GROOVE ======
  == MAJOR GROOVE ==          ========DNA=============
  ======DNA========

3. LEUCINE ZIPPER              4. HELIX-LOOP-HELIX (HLH)

    L  L  L  L                        ====
    |  |  |  |                       /    \
    ====  ====  (dimerization)      | loop |
    /        \                       \    /
   ====    ====  (basic region)       ====
  ------  ------                    --------
  = MAJOR GROOVE =                  = MAJOR GROOVE =
  =====DNA========                  ======DNA=======
```

### Nucleosomes

The nucleosome is the fundamental unit of chromatin packaging:

- **Core particle**: ~147 bp of DNA wrapped ~1.65 turns around an octamer of histone proteins (H2A, H2B, H3, H4 x 2 each)
- **Linker DNA**: 20-80 bp connecting adjacent nucleosomes
- **Linker histone** (H1): stabilizes higher-order chromatin structure
- DNA bends sharply, with the minor groove facing inward at contact points
- Nucleosome positioning is partly sequence-dependent (AA/TT dinucleotides prefer certain rotational positions)

In [ ]:
# Common transcription factor binding motifs
import re

# IUPAC ambiguity codes
IUPAC = {
    'A': 'A', 'C': 'C', 'G': 'G', 'T': 'T',
    'R': '[AG]', 'Y': '[CT]', 'S': '[GC]', 'W': '[AT]',
    'K': '[GT]', 'M': '[AC]', 'B': '[CGT]', 'D': '[AGT]',
    'H': '[ACT]', 'V': '[ACG]', 'N': '[ACGT]',
}


def iupac_to_regex(motif):
    """Convert IUPAC motif to regex pattern."""
    return ''.join(IUPAC.get(c, c) for c in motif.upper())


def find_binding_sites(sequence, motif_iupac):
    """Find all occurrences of a TF binding motif in a DNA sequence."""
    seq = sequence.upper()
    pattern = iupac_to_regex(motif_iupac)
    
    # Search forward strand
    forward_hits = [(m.start(), m.group(), '+') for m in re.finditer(pattern, seq)]
    
    # Search reverse strand
    rev_seq = reverse_complement(seq)
    reverse_hits = [(len(seq) - m.end(), m.group(), '-') for m in re.finditer(pattern, rev_seq)]
    
    return forward_hits + reverse_hits


# Example: search for TATA box (TATAWAAR) and E-box (CANNTG)
promoter = "ATGCATATAAAAGGCTTGACGCACGTGTTAAAGGATATATAAATCACCTGACC"

motifs = {
    'TATA box': 'TATAWAAR',
    'E-box':    'CANNTG',
}

print(f"Sequence: {promoter}\n")
for name, motif in motifs.items():
    hits = find_binding_sites(promoter, motif)
    print(f"{name} ({motif}): {len(hits)} hit(s)")
    for pos, match, strand in hits:
        print(f"  Position {pos} ({strand}): {match}")

In [ ]:
# Nucleosome positioning signal analysis
import numpy as np

def dinucleotide_frequency(sequence, window=20):
    """
    Calculate sliding-window dinucleotide frequencies.
    
    AA/TT dinucleotides show ~10 bp periodicity in nucleosome-bound DNA.
    """
    seq = sequence.upper()
    n = len(seq)
    
    # Count AA+TT dinucleotides in each window
    at_signal = []
    positions = []
    for i in range(n - window):
        win = seq[i:i+window]
        count = 0
        for j in range(len(win) - 1):
            dinuc = win[j:j+2]
            if dinuc in ('AA', 'TT'):
                count += 1
        at_signal.append(count / (window - 1))
        positions.append(i)
    
    return positions, at_signal


# Generate a nucleosome-positioning sequence (artificial periodic AA/TT)
import random
random.seed(42)

# Create 200 bp with periodic AA every ~10 bp
nuc_seq = list('GCGCGCGCGC' * 20)  # 200 bp background
for i in range(0, 200, 10):
    if i + 1 < 200:
        nuc_seq[i] = 'A'
        nuc_seq[i+1] = 'A'
nuc_seq = ''.join(nuc_seq)

positions, signal = dinucleotide_frequency(nuc_seq, window=10)

print(f"Sequence length: {len(nuc_seq)} bp")
print(f"AA+TT frequency shows ~10 bp periodicity (nucleosome positioning signal)")
print(f"\nFirst 50 bp: {nuc_seq[:50]}")
print(f"Signal (first 10 windows): {[f'{s:.2f}' for s in signal[:10]]}")

---

## 9. Analyzing Nucleic Acid Structures from PDB

Nucleic acid residues in PDB files use specific naming conventions:

| Residue name | Meaning |
|-------------|--------|
| DA, DT, DG, DC | DNA nucleotides |
| A, U, G, C | RNA nucleotides |

The sugar-phosphate backbone atoms include: P, OP1, OP2, O5', C5', C4', O4', C1', C2', O2' (RNA), C3', O3'

Key base atoms for geometry analysis:
- **N1** (pyrimidines) / **N9** (purines): glycosidic bond atom
- **C6** (pyrimidines) / **C8** (purines): used for base pair geometry
- **C1'**: sugar atom attached to the base

In [ ]:
from Bio.PDB import PDBParser, PDBList
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Download a DNA-protein complex: EcoRI endonuclease bound to DNA (1ERI)
pdbl = PDBList()
pdb_file = pdbl.retrieve_pdb_file('1BNA', pdir='pdb_files', file_format='pdb')
print(f"Downloaded: {pdb_file}")
# 1BNA = Dickerson dodecamer, a classic B-DNA crystal structure (CGCGAATTCGCG)

In [ ]:
# Parse and analyze the DNA structure
parser = PDBParser(QUIET=True)
structure = parser.get_structure('dna', pdb_file)

DNA_RESIDUES = {'DA', 'DT', 'DG', 'DC', 'DU'}
RNA_RESIDUES = {'A', 'U', 'G', 'C'}

print("=== Nucleic Acid Structure Analysis ===")
for model in structure:
    print(f"\nModel {model.id}:")
    for chain in model:
        residues = [r for r in chain if r.id[0] == ' ']
        if not residues:
            continue
        
        # Determine type
        first_resname = residues[0].get_resname().strip()
        if first_resname in DNA_RESIDUES:
            na_type = 'DNA'
        elif first_resname in RNA_RESIDUES:
            na_type = 'RNA'
        else:
            na_type = 'Protein'
        
        # Extract sequence
        seq = ''
        for r in residues:
            resname = r.get_resname().strip()
            if resname in DNA_RESIDUES:
                seq += resname[1]  # Remove 'D' prefix
            elif resname in RNA_RESIDUES:
                seq += resname
            else:
                seq += 'X'
        
        print(f"  Chain {chain.id}: {na_type}, {len(residues)} residues")
        print(f"    Sequence: 5'-{seq}-3'")

In [ ]:
# Measure base pair parameters from the structure
def measure_base_pair_distance(residue1, residue2):
    """
    Measure the C1'-C1' distance between two nucleotides.
    In a canonical base pair, this distance is ~10.4 A.
    """
    if "C1'" in residue1 and "C1'" in residue2:
        c1_1 = residue1["C1'"].get_vector().get_array()
        c1_2 = residue2["C1'"].get_vector().get_array()
        return np.linalg.norm(c1_1 - c1_2)
    return None


def measure_rise(residue1, residue2):
    """
    Measure the rise (vertical distance) between consecutive base pairs.
    Uses the P atom as reference. B-DNA rise ~ 3.4 A.
    """
    if 'P' in residue1 and 'P' in residue2:
        p1 = residue1['P'].get_vector().get_array()
        p2 = residue2['P'].get_vector().get_array()
        return np.linalg.norm(p1 - p2)
    return None


# Measure distances along chain A
model = structure[0]
chains = list(model.get_chains())
chain_a = chains[0]
chain_b = chains[1] if len(chains) > 1 else None

residues_a = [r for r in chain_a if r.id[0] == ' ']

# Measure P-P distances (rise between consecutive nucleotides)
print("Consecutive P-P distances along chain A (rise indicator):")
print(f"{'Step':>5s} {'Res_i':>6s} {'Res_j':>6s} {'P-P dist (A)':>13s}")
print("-" * 35)
for i in range(len(residues_a) - 1):
    d = measure_rise(residues_a[i], residues_a[i+1])
    if d is not None:
        name_i = residues_a[i].get_resname().strip()
        name_j = residues_a[i+1].get_resname().strip()
        print(f"{i+1:5d} {name_i:>6s} {name_j:>6s} {d:13.2f}")

In [ ]:
# Measure complementary base pair C1'-C1' distances
# In 1BNA, chain A and chain B form the two strands of the double helix
if chain_b is not None:
    residues_b = [r for r in chain_b if r.id[0] == ' ']
    
    # The strands are antiparallel, so residue i in chain A pairs with
    # residue (N-i+1) in chain B
    n = min(len(residues_a), len(residues_b))
    
    print("Base pair C1'-C1' distances (expected ~10.4 A for B-DNA):")
    print(f"{'Chain A':>10s} {'Chain B':>10s} {\"C1'-C1' (A)\":>13s}")
    print("-" * 37)
    for i in range(n):
        j = n - 1 - i  # Antiparallel pairing
        d = measure_base_pair_distance(residues_a[i], residues_b[j])
        if d is not None:
            name_a = residues_a[i].get_resname().strip()
            name_b = residues_b[j].get_resname().strip()
            bp_ok = "OK" if 9.5 < d < 11.5 else "unusual"
            print(f"{name_a:>10s} {name_b:>10s} {d:13.2f}  {bp_ok}")
else:
    print("Only one chain found; cannot measure base pair distances.")

In [ ]:
# Backbone torsion angles
# DNA backbone is described by 6 torsion angles: alpha, beta, gamma, delta, epsilon, zeta
# Plus the glycosidic angle chi

def dihedral_angle(p1, p2, p3, p4):
    """Calculate dihedral angle (degrees) from four coordinate arrays."""
    b1 = p2 - p1
    b2 = p3 - p2
    b3 = p4 - p3
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    n1 /= np.linalg.norm(n1)
    n2 /= np.linalg.norm(n2)
    m1 = np.cross(n1, b2 / np.linalg.norm(b2))
    return np.degrees(np.arctan2(np.dot(m1, n2), np.dot(n1, n2)))


def get_coord(residue, atom_name):
    """Safely get atom coordinates from a residue."""
    if atom_name in residue:
        return residue[atom_name].get_vector().get_array()
    return None


def measure_sugar_pucker(residue):
    """
    Estimate sugar pucker from the C1'-C2'-C3'-C4' torsion.
    C2'-endo: B-form DNA (~140-180 deg)
    C3'-endo: A-form RNA (~0-40 deg)
    """
    c1 = get_coord(residue, "C1'")
    c2 = get_coord(residue, "C2'")
    c3 = get_coord(residue, "C3'")
    c4 = get_coord(residue, "C4'")
    
    if any(x is None for x in [c1, c2, c3, c4]):
        return None, 'unknown'
    
    nu2 = dihedral_angle(c1, c2, c3, c4)
    
    # Simplified classification
    if -50 < nu2 < 50:
        pucker = "C3'-endo (A-form)"
    else:
        pucker = "C2'-endo (B-form)"
    
    return nu2, pucker


# Analyze sugar pucker for each nucleotide
print("Sugar pucker analysis:")
print(f"{'Residue':>10s} {'nu2 (deg)':>10s} {'Pucker':>20s}")
print("-" * 44)
for res in residues_a:
    name = res.get_resname().strip()
    num = res.id[1]
    nu2, pucker = measure_sugar_pucker(res)
    if nu2 is not None:
        print(f"{name + str(num):>10s} {nu2:10.1f} {pucker:>20s}")

---

## 10. Visualization of Nucleic Acid Structures

### nglview in Jupyter

In [ ]:
# Interactive 3D visualization of DNA structure
try:
    import nglview as nv
    
    view = nv.show_biopython(structure)
    view.clear_representations()
    
    # Show DNA as cartoon + base pairs
    view.add_cartoon(color='chainid')
    view.add_base(color='resname')  # Show bases as flat rings
    view.add_licorice('not polymer', color='element')  # Ligands/ions
    
    view.background = 'white'
    print("Interactive DNA structure viewer (drag to rotate, scroll to zoom):")
    display(view)

except ImportError:
    print("nglview is not installed. Install with:")
    print("  pip install nglview")
    print("  jupyter nbextension enable nglview --py --sys-prefix")

### PyMOL Commands for Nucleic Acids

```python
# Load DNA structure
fetch 1BNA

# Cartoon representation for nucleic acids
show cartoon         # Shows sugar-phosphate backbone as ribbon
show sticks, (resn DA+DT+DG+DC) and (name N1+N2+N3+N4+N6+N7+N9+C2+C4+C5+C6+C8+O2+O4+O6)

# Color by base type
color red, resn DA    # Adenine = red
color blue, resn DT   # Thymine = blue
color green, resn DG  # Guanine = green
color yellow, resn DC # Cytosine = yellow

# Show hydrogen bonds between base pairs
distance hbonds, (chain A), (chain B), 3.5, mode=2

# Measure groove widths
# (measure P-P distances across the groove)

# Surface representation
show surface
set transparency, 0.4

# Save publication-quality image
ray 2400, 2400
png dna_structure.png, dpi=300
```

### Other Specialized Tools

| Tool | Purpose | URL |
|------|---------|-----|
| 3DNA / x3dna | Complete DNA/RNA structural analysis (parameters, rebuild) | http://x3dna.org |
| DSSR | RNA structure annotation (pairs, motifs, topology) | http://x3dna.org/dssr |
| Curves+ | DNA helical parameters and groove dimensions | http://curvesplus.bsc.es |
| RNApdbee | RNA secondary structure from 3D coordinates | http://rnapdbee.cs.put.poznan.pl |
| VARNA | RNA secondary structure visualization | http://varna.lri.fr |

---

## 11. Modified Nucleotides

Nucleic acids contain many biologically important modifications:

### DNA Modifications

| Modification | Notation | Role |
|-------------|----------|------|
| 5-methylcytosine | m5C | Epigenetic gene silencing; CpG islands |
| 5-hydroxymethylcytosine | hm5C | Intermediate in demethylation |
| N6-methyladenine | m6A | Bacterial restriction-modification |

### RNA Modifications

Over 170 RNA modifications are known. The most common:

| Modification | Symbol | Found In | Function |
|-------------|--------|----------|----------|
| Pseudouridine | PSU | tRNA, rRNA, snRNA | Stabilizes structure |
| N6-methyladenosine | m6A | mRNA | Regulates mRNA metabolism |
| 2'-O-methylation | Nm | rRNA, snRNA | Protects from nucleases |
| Dihydrouridine | D | tRNA (D-loop) | Increases flexibility |
| Inosine | I | tRNA anticodon | Wobble pairing (pairs with A, C, U) |

---

## 12. Exercises

### Exercise 1: GC Content and Melting Temperature

Write a function that takes a DNA sequence and calculates:
1. GC content (%)
2. Melting temperature (using both the simple formula and nearest-neighbor model)
3. Number of hydrogen bonds in the duplex

Test with these sequences:
- `ATATATATATATATAT` (AT-rich)
- `GCGCGCGCGCGCGCGC` (GC-rich)
- `AGTCAGTCAGTCAGTC` (balanced)

In [ ]:
# Exercise 1: Your solution here
# Hint: Use the gc_content(), melting_temperature(), and nearest_neighbor_dG()
# functions defined above as starting points.


### Exercise 2: Restriction Enzyme Sites

Write a function to find all occurrences of restriction enzyme recognition sites in a DNA sequence. Search both strands. Test with:

| Enzyme | Recognition Site | Cut Pattern |
|--------|-----------------|-------------|
| EcoRI | GAATTC | G^AATTC |
| BamHI | GGATCC | G^GATCC |
| HindIII | AAGCTT | A^AGCTT |
| EcoRV | GATATC | GAT^ATC |

In [ ]:
# Exercise 2: Your solution here
test_dna = "ATGCGAATTCGATGGATCCAAGCTTGATATCGCAATTCGGATCCATG"

# Hint: For each enzyme, search for the recognition sequence on both strands.
# Remember that palindromic restriction sites read the same on both strands
# (5'->3'), so finding them on one strand is often sufficient.


### Exercise 3: RNA Secondary Structure Analysis

Given these RNA sequences, predict their secondary structure using the Nussinov algorithm and analyze the results:

1. `GGGAAACCC` (simple hairpin)
2. `GGGCCCAAAGGGCCC` (two hairpins)
3. `GGGGAAAACCCCUUUUAAAACCCC` (more complex)

For each, report: number of base pairs, hairpin loops, and paired fraction.

In [ ]:
# Exercise 3: Your solution here
# Use nussinov_fold() and classify_structure_elements() from Section 6.
rna_sequences = [
    "GGGAAACCC",
    "GGGCCCAAAGGGCCC",
    "GGGGAAAACCCCUUUUAAAACCCC",
]

# For each sequence, predict structure and analyze


### Exercise 4: DNA-Protein Complex Analysis

Download PDB entry **1ERI** (EcoRI endonuclease bound to DNA). Analyze the complex:

1. Identify the protein chains and the DNA chains
2. Find all protein residues within 4 Angstroms of any DNA atom
3. Classify these contact residues by amino acid type
4. Which amino acids are most commonly found at the DNA interface?

In [ ]:
# Exercise 4: Your solution here
# Hint:
# 1. Download 1ERI with PDBList
# 2. Parse and identify DNA vs protein chains
# 3. For each protein residue, check if any atom is within 4 A
#    of any DNA atom (use Bio.PDB.NeighborSearch for efficiency)
# 4. Count amino acid types at the interface


### Exercise 5: Helix Parameter Comparison

Download two DNA structures:
- **1BNA**: B-DNA (Dickerson dodecamer)
- **440D**: A-DNA crystal structure

For each structure:
1. Measure consecutive P-P distances along one strand
2. Calculate the mean and standard deviation of P-P distances
3. Analyze sugar pucker for each nucleotide
4. Compare: does the P-P distance and sugar pucker match expectations for A-form vs B-form?

In [ ]:
# Exercise 5: Your solution here
# Hint:
# Use measure_rise() and measure_sugar_pucker() functions from Section 9.
# Expected P-P distances: B-DNA ~7 A, A-DNA ~6 A


### Exercise 6: Z-DNA Detection

Z-DNA tends to form in regions with alternating purine-pyrimidine sequences. Write a function that scans a DNA sequence and scores each window for Z-DNA forming potential based on:

1. Fraction of alternating purine-pyrimidine dinucleotides
2. Presence of CG or CA/TG dinucleotide repeats

Test on a sequence containing an embedded (CG)n repeat.

In [ ]:
# Exercise 6: Your solution here
PURINES = set('AG')
PYRIMIDINES = set('CT')

def z_dna_potential(sequence, window=12):
    """
    Score each window for Z-DNA forming potential.
    Returns list of (position, score) tuples.
    """
    # Your code here
    pass

# Test sequence with embedded CG repeat
test_seq = "AATTTTT" + "CGCGCGCGCGCG" + "AAATTTAAA"
# z_scores = z_dna_potential(test_seq)


---

## Summary

| Topic | Key Concepts |
|-------|-------------|
| Nucleotide building blocks | Base (purine/pyrimidine) + sugar (ribose/deoxyribose) + phosphate |
| Watson-Crick pairing | A-T (2 H-bonds), G-C (3 H-bonds); Chargaff's rules |
| DNA helical forms | A (dehydrated, RNA), B (physiological), Z (left-handed, alternating pur-pyr) |
| Grooves | Major groove: protein recognition; Minor groove: AT-rich binding |
| Base stacking | Major stabilizing force; nearest-neighbor model for thermodynamics |
| RNA secondary structure | Stems, hairpin loops, bulges, internal loops, junctions; dot-bracket notation |
| RNA folding prediction | MFE approach, Nussinov (maximize pairs), Zuker (full thermodynamics), ViennaRNA |
| RNA tertiary structure | Non-WC base pairs, A-minor motifs, kissing loops, pseudoknots |
| DNA-protein interactions | Major groove recognition by TFs (HTH, zinc finger, leucine zipper); nucleosomes |
| PDB analysis | DNA residues (DA/DT/DG/DC), backbone torsions, sugar pucker, base pair geometry |
| Modified nucleotides | m5C (epigenetics), pseudouridine (RNA stability), m6A (mRNA regulation) |

---

## Resources

- [RCSB PDB](https://www.rcsb.org/) -- Protein Data Bank (includes DNA/RNA structures)
- [Nucleic Acid Database (NDB)](https://ndbserver.rutgers.edu/) -- Specialized NA structure database
- [ViennaRNA](https://www.tbi.univie.ac.at/RNA/) -- RNA secondary structure prediction
- [3DNA / DSSR](http://x3dna.org/) -- Nucleic acid structural analysis
- [Rfam](https://rfam.org/) -- RNA families database
- [Curves+](http://curvesplus.bsc.es/) -- DNA/RNA helix parameter analysis
- [VARNA](http://varna.lri.fr/) -- RNA secondary structure visualization
- [mfold](http://www.unafold.org/) -- RNA/DNA folding server

---[< Previous: Protein Structure Analysis](../07_Protein_Structure/01_protein_structure.ipynb) | [Tier 2: Core Bioinformatics Overview](../README.md) | [Next: Chromatogram Analysis: Sanger Sequencing in Pra... >](../09_Chromatogram_Analysis/01_chromatogram_analysis.ipynb)---